In this note I fully compare the results obtained by optimizing for the commutator cost function:

$$
C_{ref} = \sum_i || [H, S_i] |ref \rangle ||^2
$$

Here "ref" can be "HF" or "FCI". $S_i$ is a generalized seniority operator:

$ S_i = U^\dagger (a n_{i \alpha} + b n_{i \beta} + c n_{i \alpha} n_{i \beta}) U $

where $U$ is an orbital rotation and $a^2 + b^2 + c^2 = 1$. When (a, b, c) is proportional to (1, 1, -2), we call $S_i$ seniority.

 For a given Hamiltonian (one fixed geometry), we can compare the following quasisymmetries:

1. Seniorities in canonical orbital basis
2. Seniorities in orbital basis optimized for $C_{FCI}$ without changing the $(a, b, c)$ coefficients
3. Same but with optimizable coefficients
4. Same as pt 2 but for $C_{HF}$
5. Same as pt 3 but for $C_{HF}$

Each optimization procedure was run multiple times with different initial guesses, I'll take the best one.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import sys
import pyscf
import scipy
import ffsim
import os
import json

from itertools import combinations

sys.path.insert(0, "..")

import optimize
import xs_to_metrics

In [2]:
results_folder = "../outputs/H4_linear/1.089//"

In [3]:
xs_data_filenames = [s for s in list(os.walk(results_folder))[0][2]
                    if s[-9:] == "x_opt.txt"]

print(xs_data_filenames)

['20260512_161402_x_opt.txt', '20260521_163502_x_opt.txt', '20260512_160540_x_opt.txt', '20260521_162527_x_opt.txt']


In [4]:
metrics_tables = []
specs = []

for current_filename in xs_data_filenames:

    with open(results_folder + current_filename, "r") as fp:
        experiment_specs = fp.readline()
        spec_string = experiment_specs.replace("'", '"').replace("True", "true").replace("False", "false")
        spec_string = spec_string[:-1]
        # print(spec_string)
        # print(spec_string[167])
        specs.append(json.loads(spec_string))

    metrics_tables.append(pd.read_csv(results_folder + current_filename + "_metrics.txt", sep=" "))

In [5]:
specs

[{'molpath': 'hamiltonians/H4_linear_d1.0890.chk',
  'initialguesses': 'x0_4_perturb_all_0_20260512_160423.txt',
  'cost_function': 'commutator',
  'verbose': True,
  'reference': 'hf'},
 {'molpath': 'hamiltonians/H4_linear_d1.0890.chk',
  'initialguesses': 'initial_points/x0_4_perturb_all_0_20260512_160423.txt',
  'cost_function': 'commutator',
  'verbose': False,
  'reference': 'fci',
  'seniority': True},
 {'molpath': 'hamiltonians/H4_linear_d1.0890.chk',
  'initialguesses': 'x0_4_perturb_all_0_20260512_160423.txt',
  'cost_function': 'commutator',
  'verbose': True,
  'reference': 'fci'},
 {'molpath': 'hamiltonians/H4_linear_d1.0890.chk',
  'initialguesses': 'initial_points/x0_4_perturb_all_0_20260512_160423.txt',
  'cost_function': 'commutator',
  'verbose': False,
  'reference': 'hf',
  'seniority': True}]

In [6]:
rows = []
for i, spec in enumerate(specs):
    if spec["reference"] == "fci":
        row = metrics_tables[i].iloc[np.argmin(metrics_tables[i]["C_fci"])]
        title = "FCI"
    elif spec["reference"] == "hf":
        row = metrics_tables[i].iloc[np.argmin(metrics_tables[i]["C_hf"])]
        title = "HF"
    else:
        raise ValueError()

    condition = (("seniority" not in spec.keys()) 
                 or ("seniority" in spec.keys() and spec["seniority"] == False))
    
    if condition:
        title += ", abc optimized"
    else:
        title += ", abc=(1, 1, -2)"

    row.name = title
    rows.append(row)

In [10]:
molpath = "../" + specs[0]["molpath"]
mol = pyscf.lib.chkfile.load_mol(molpath)
mf = pyscf.scf.RHF(mol)
mf.update_from_chk(molpath)
moldata = ffsim.MolecularData.from_scf(mf)

x = np.zeros(len(list(combinations(range(moldata.norb), 2))) + 2)
x[-2:] = optimize.SENIORITY_ANGLES

phi1, phi2 = x[-2], x[-1]
a_opt = np.sin(phi1) * np.cos(phi2)
b_opt = np.sin(phi1) * np.sin(phi2)
c_opt = np.cos(phi1)

commutator_fci = optimize.commutator_cost(moldata, "fci")
commutator_hf = optimize.commutator_cost(moldata, "hf")
variance_fci = optimize.variance_cost(moldata, "fci")
variance_hf = optimize.variance_cost(moldata, "hf")

energies, sectors_data = xs_to_metrics.sector_partitioning_metrics(moldata, np.eye(moldata.norb), 
                                                                   a_opt, b_opt, c_opt)
costs_and_bc = np.array([variance_fci(x), variance_hf(x),
                         commutator_fci(x), commutator_hf(x),
                         b_opt / a_opt, c_opt / a_opt,
                         energies["Decoupled"] - energies["FCI"],
                         energies["Born-Oppenheimer"] - energies["FCI"],
                         sectors_data["K_en"], sectors_data["K_overlap"],
                         sectors_data["sectors_en"], sectors_data["sectors_overlap"]]).real

[['_', '*'], ['^', 'v']]
(0, 0, 0, 0)
(0, 0, 1, 1)
(0, 1, 0, 1)
(0, 1, 1, 0)
(1, 0, 0, 1)
(1, 0, 1, 0)
(1, 1, 0, 0)
(1, 1, 1, 1)
[(0, 0, 0, 0), (1, 1, 1, 1)]
[(0, 0, 0, 0), (0, 0, 1, 1), (0, 1, 0, 1), (0, 1, 1, 0), (1, 0, 0, 1), (1, 0, 1, 0), (1, 1, 0, 0), (1, 1, 1, 1)]


In [11]:
canonical_sen = pd.Series(data=costs_and_bc, index=rows[0].index, name="MO")

metrics_summary = pd.concat([canonical_sen] + rows, axis=1).T
metrics_summary = metrics_summary.sort_index()
metrics_summary

,V_fci,V_hf,C_fci,C_hf,b,c,E_dec-E_fci,E_bo-E_fci,K_en,K_overlap,Sectors_en,Sectors_overlap
"FCI, abc optimized",0.011382,1.189206e-03,0.009547,0.004518,1.000004,-1.917420,0.016189,0.016189,24.0,7.0,8.0,6.0
"FCI, abc=(1, 1, -2)",0.011440,1.220297e-03,0.009753,0.004442,1.000000,-2.000000,0.016161,0.016161,24.0,7.0,8.0,6.0
"HF, abc optimized",0.010093,7.078587e-05,0.010137,0.003954,1.000019,-1.975792,0.016100,0.016100,24.0,5.0,8.0,4.0
"HF, abc=(1, 1, -2)",0.010137,7.195784e-05,0.010240,0.003969,1.000000,-2.000000,0.016099,0.016099,24.0,5.0,8.0,4.0
MO,0.017462,-5.546678e-32,0.051473,0.020530,1.000000,-2.000000,0.040705,0.040705,32.0,5.0,8.0,2.0


In [38]:
h_linop = ffsim.linear_operator(moldata.hamiltonian,
                                    norb=moldata.norb,
                                    nelec=moldata.nelec)

e_0, v_0 = scipy.sparse.linalg.eigsh(h_linop, which="SA", k=1)
print(e_0)

[-1.87116806]


In [39]:
metrics_summary["E_dec"] - e_0

FCI, abc optimized     0.008902
FCI, abc optimized     0.001437
FCI, abc=(1, 1, -2)    0.001437
HF, abc optimized      0.008579
MO                     0.008579
Name: E_dec, dtype: float64

In [19]:
column_names = [r"$V_{\mathrm{FCI}}$", r"$V_{\mathrm{FCI}}$", 
                r"$C_{\mathrm{FCI}}$", r"$C_{\mathrm{FCI}}$",
                "b", "c", r"$E_{\mathrm{dec}}$", r"$E_{\mathrm{BO}}$",
                r"$K_{E}$", r"$K_{\mathrm{overlap}}$"]

print(metrics_summary.to_latex())

\begin{tabular}{lrrrrrrrrrr}
\toprule
 & V_fci & V_hf & C_fci & C_hf & b & c & E_dec & E_bo & K_en & K_overlap \\
\midrule
FCI, abc optimized & 0.041588 & 0.017865 & 0.001439 & 0.002065 & 1.000014 & -1.991615 & -1.885213 & -1.885213 & 29.000000 & 7.000000 \\
FCI, abc optimized & 1.909553 & 0.999842 & 0.003807 & 0.015951 & -0.999969 & 0.000073 & -1.844108 & -1.861411 & 25.000000 & 17.000000 \\
FCI, abc=(1, 1, -2) & 0.041694 & 0.017909 & 0.001443 & 0.002069 & 1.000000 & -2.000000 & -1.885214 & -1.885214 & 29.000000 & 7.000000 \\
HF, abc optimized & 0.042068 & 0.000300 & 0.005315 & 0.001226 & 0.999780 & -1.994606 & -1.873256 & -1.873256 & 29.000000 & 5.000000 \\
HF, abc=(1, 1, -2) & 0.042139 & 0.000300 & 0.005326 & 0.001227 & 1.000000 & -2.000000 & -1.873248 & -1.873248 & 29.000000 & 5.000000 \\
MO & 0.114067 & -0.000000 & 0.182144 & 0.038886 & 1.000000 & -2.000000 & -1.845397 & -1.867659 & 36.000000 & 10.000000 \\
\bottomrule
\end{tabular}

